# Mask R-CNN for Individual Non-Musanga Crown Delineation

This notebook trains a **Mask R-CNN instance-segmentation model** to delineate individual non-Musanga tree crowns from UAV imagery.

## Inputs

- high-resolution RGB orthomosaic;
- DSM and, when available, DTM;
- canopy-height model derived as `CHM = DSM − DTM`;
- manually digitized crown polygons;
- existing U-Net Musanga mask, used to exclude Musanga-dominated areas.

## Target

Each non-Musanga tree crown is treated as a separate instance. The notebook therefore learns both:

1. where a crown is located;
2. the individual mask of that crown.

## Important limitation

The existing non-Musanga polygons appear conservative and may represent crown cores rather than complete crowns. The notebook therefore supports two training-label modes:

- `ORIGINAL`: train directly from the supplied polygons;
- `CHM_EXPANDED`: expand each polygon toward plausible CHM/RGB crown boundaries before training.

Start with `QUICK_TEST = True`. Once the workflow runs correctly, switch it to `False` for the full experiment.


## BLOCK 01 — Install and import packages

In [1]:

# ======================================================================================
# BLOCK 01 — INSTALL AND IMPORT PACKAGES
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 01 — INSTALL AND IMPORT PACKAGES")
print("=" * 100)

!pip -q install geopandas rasterio shapely pyogrio rtree scikit-image scipy

from pathlib import Path
import copy
import json
import math
import os
import random
import shutil
import time
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torchvision

from PIL import Image
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from rasterio.windows import Window
from rasterio.warp import reproject
from scipy import ndimage as ndi
from shapely.geometry import box, shape
from skimage import morphology, segmentation
from skimage.feature import peak_local_max
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import (
    MaskRCNN_ResNet50_FPN_Weights,
    maskrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())



BLOCK 01 — INSTALL AND IMPORT PACKAGES
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
CUDA available: True


## BLOCK 02 — Mount Drive and define project folders

In [2]:

# ======================================================================================
# BLOCK 02 — ROBUST GOOGLE DRIVE MOUNT
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 02 — ROBUST GOOGLE DRIVE MOUNT")
print("=" * 100)

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"

if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

PROJECT_ROOT = MYDRIVE / "Colab Notebooks" / "Musanga"
WORK_DIR = PROJECT_ROOT / "work"
DATA_DIR = PROJECT_ROOT / "data"

MODEL_DIR = WORK_DIR / "mask_rcnn_non_musanga"
CACHE_DIR = WORK_DIR / "mask_rcnn_patch_cache"
OUTPUT_DIR = WORK_DIR / "mask_rcnn_non_musanga_outputs"

for directory in [
    MODEL_DIR,
    CACHE_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)



BLOCK 02 — ROBUST GOOGLE DRIVE MOUNT
Google Drive already mounted.
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Musanga
MODEL_DIR: /content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_patch_cache
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga_outputs


## BLOCK 03 — Configuration

In [3]:

# ======================================================================================
# BLOCK 03 — CONFIGURATION
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 03 — CONFIGURATION")
print("=" * 100)

# ----------------------------------------------------------------------
# RUN MODE
# ----------------------------------------------------------------------

QUICK_TEST = False

# Change to False only after the quick test runs correctly.
RUN_TRAINING = False

RANDOM_SEED = 42

# ----------------------------------------------------------------------
# INPUT PATHS
# ----------------------------------------------------------------------

DATA_DIR = (
    Path("/content/drive/MyDrive/Colab Notebooks/Musanga/data")
)

RGB_PATH = (
    DATA_DIR
    / "Carlier2020_Aug24_25_survey_transparent_mosaic_group1.tif"
)

DSM_PATH = (
    DATA_DIR
    / "Carlier2020_Aug24_25_survey_dsm.tif"
)

# No DTM is visible in the screenshot.
# The notebook will estimate a local ground surface from the DSM.
DTM_PATH = None

REFERENCE_CROWNS_PATH = (
    DATA_DIR
    / "Sample_Musanga_others_2020_2024_complet.shp"
)

MUSANGA_MASK_PATH = (
    Path(
        "/content/drive/MyDrive/Colab Notebooks/Musanga/work/"
        "whole_orthomosaic_inference/"
        "musanga_binary_mask_threshold_0.008.tif"
    )
)

DTM_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Musanga/data/"
    "Yoko_DEMpred.tif"
)

# ----------------------------------------------------------------------
# LABEL MODE
#
# ORIGINAL:
#   Use supplied non-Musanga polygons unchanged.
#
# CHM_EXPANDED:
#   Expand conservative crown-core polygons using local CHM watershed.
# ----------------------------------------------------------------------

LABEL_MODE = "CHM_EXPANDED"

# ----------------------------------------------------------------------
# PATCH AND MODEL SETTINGS
# ----------------------------------------------------------------------

PATCH_SIZE = 512
PATCH_OVERLAP = 64

# RGB + CHM = four channels.
N_INPUT_CHANNELS = 4

BATCH_SIZE = 2 if QUICK_TEST else 4
NUM_WORKERS = 0

EPOCHS = 2 if QUICK_TEST else 20
LEARNING_RATE = 0.0025
WEIGHT_DECAY = 0.0001

MAX_TRAIN_PATCHES = 300 if QUICK_TEST else None
MAX_VALIDATION_PATCHES = 100 if QUICK_TEST else None
MAX_TEST_PATCHES = 100 if QUICK_TEST else None

# Detection threshold used only for evaluation/inference.
SCORE_THRESHOLD = 0.50
MASK_THRESHOLD = 0.50

# ----------------------------------------------------------------------
# CROWN FILTERS
# ----------------------------------------------------------------------

MIN_CROWN_AREA_M2 = 1.0
MAX_CROWN_AREA_M2 = 500.0
MIN_TREE_HEIGHT_M = 2.0

# Conservative polygon expansion.
MAX_EXPANSION_DISTANCE_M = 3.0
MIN_EXPANDED_IOU_WITH_CORE = 0.40

# ----------------------------------------------------------------------
# SPATIAL SPLIT
# ----------------------------------------------------------------------

SPATIAL_BLOCK_SIZE_M = 150.0
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

# ----------------------------------------------------------------------
# REPRODUCIBILITY
# ----------------------------------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("QUICK_TEST:", QUICK_TEST)
print("RUN_TRAINING:", RUN_TRAINING)
print("LABEL_MODE:", LABEL_MODE)
print("DEVICE:", DEVICE)



BLOCK 03 — CONFIGURATION
QUICK_TEST: False
RUN_TRAINING: False
LABEL_MODE: CHM_EXPANDED
DEVICE: cuda


## BLOCK 04 — Discover and validate input files

In [4]:

# ======================================================================================
# BLOCK 04 — DISCOVER AND VALIDATE INPUT FILES
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 04 — DISCOVER AND VALIDATE INPUT FILES")
print("=" * 100)

SEARCH_ROOTS = [
    Path("/content/musanga_training/sources"),
    DATA_DIR,
    PROJECT_ROOT,
    WORK_DIR,
]

def find_first(patterns):
    matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for pattern in patterns:
            matches.extend(root.glob(pattern))
    return sorted(matches)[0] if matches else None

if DSM_PATH is None:
    DSM_PATH = find_first([
        "**/*DSM*.tif",
        "**/*dsm*.tif",
        "**/*surface*.tif",
    ])

if DTM_PATH is None:
    DTM_PATH = find_first([
        "**/*DTM*.tif",
        "**/*dtm*.tif",
        "**/*terrain*.tif",
    ])

if REFERENCE_CROWNS_PATH is None:
    REFERENCE_CROWNS_PATH = find_first([
        "**/*tree*.shp",
        "**/*crown*.shp",
        "**/*classified*.shp",
        "**/*tree*.gpkg",
        "**/*crown*.gpkg",
    ])

if MUSANGA_MASK_PATH is None:
    MUSANGA_MASK_PATH = find_first([
        "**/musanga_binary_mask_threshold_*.tif",
    ])

required = {
    "RGB orthomosaic": RGB_PATH,
    "DSM": DSM_PATH,
    "Reference crowns": REFERENCE_CROWNS_PATH,
    "Musanga mask": MUSANGA_MASK_PATH,
}

missing = [
    name
    for name, path in required.items()
    if path is None or not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required inputs:\n"
        f"{missing}\n\n"
        "Set the correct paths manually in Block 03."
    )

RGB_PATH = Path(RGB_PATH)
DSM_PATH = Path(DSM_PATH)
DTM_PATH = Path(DTM_PATH) if DTM_PATH else None
REFERENCE_CROWNS_PATH = Path(REFERENCE_CROWNS_PATH)
MUSANGA_MASK_PATH = Path(MUSANGA_MASK_PATH)

print("RGB:", RGB_PATH)
print("DSM:", DSM_PATH)
print("DTM:", DTM_PATH)
print("Reference crowns:", REFERENCE_CROWNS_PATH)
print("Musanga mask:", MUSANGA_MASK_PATH)



BLOCK 04 — DISCOVER AND VALIDATE INPUT FILES
RGB: /content/drive/MyDrive/Colab Notebooks/Musanga/data/Carlier2020_Aug24_25_survey_transparent_mosaic_group1.tif
DSM: /content/drive/MyDrive/Colab Notebooks/Musanga/data/Carlier2020_Aug24_25_survey_dsm.tif
DTM: /content/drive/MyDrive/Colab Notebooks/Musanga/data/Yoko_DEMpred.tif
Reference crowns: /content/drive/MyDrive/Colab Notebooks/Musanga/data/Sample_Musanga_others_2020_2024_complet.shp
Musanga mask: /content/drive/MyDrive/Colab Notebooks/Musanga/work/whole_orthomosaic_inference/musanga_binary_mask_threshold_0.008.tif


## BLOCK 05 — Prepare tiled local RGB and aligned CHM

In [5]:
# ======================================================================================
# BLOCK 05 — MEMORY-SAFE TILED RGB AND PARTIAL-DTM CHM
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 05 — MEMORY-SAFE TILED RGB AND PARTIAL-DTM CHM")
print("=" * 100)

from pathlib import Path

import numpy as np
import rasterio

from rasterio.enums import Resampling
from rasterio.shutil import copy as rio_copy
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window


# ----------------------------------------------------------------------
# LOCAL PATHS
# ----------------------------------------------------------------------

LOCAL_DATA_DIR = Path(
    "/content/mask_rcnn_sources"
)

LOCAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_RGB_PATH = (
    LOCAL_DATA_DIR
    / "rgb_tiled_512.tif"
)

LOCAL_CHM_PATH = (
    LOCAL_DATA_DIR
    / "chm_partial_dtm_aligned.tif"
)

LOCAL_DTM_COVERAGE_PATH = (
    LOCAL_DATA_DIR
    / "dtm_coverage_aligned.tif"
)

LOCAL_MUSANGA_MASK_PATH = (
    LOCAL_DATA_DIR
    / "musanga_mask_aligned.tif"
)

FORCE_REBUILD_LOCAL_INPUTS = False

OUTPUT_BLOCK_SIZE = 512

CHM_NODATA = -9999.0
MASK_NODATA = 255


# ----------------------------------------------------------------------
# CREATE OR REUSE LOCALLY TILED RGB
# ----------------------------------------------------------------------

def valid_tiled_rgb(
    path,
):

    if not path.exists():
        return False

    try:

        with rasterio.open(
            path
        ) as source:

            return (
                source.count >= 3
                and source.block_shapes[0]
                == (
                    OUTPUT_BLOCK_SIZE,
                    OUTPUT_BLOCK_SIZE,
                )
            )

    except Exception:

        return False


if (
    FORCE_REBUILD_LOCAL_INPUTS
    or not valid_tiled_rgb(
        LOCAL_RGB_PATH
    )
):

    print(
        "Creating local tiled RGB copy..."
    )

    if LOCAL_RGB_PATH.exists():
        LOCAL_RGB_PATH.unlink()

    rio_copy(
        RGB_PATH,
        LOCAL_RGB_PATH,
        driver="GTiff",
        tiled=True,
        blockxsize=OUTPUT_BLOCK_SIZE,
        blockysize=OUTPUT_BLOCK_SIZE,
        compress="LZW",
        predictor=2,
        BIGTIFF="YES",
    )

else:

    print(
        "Reusing local tiled RGB."
    )


# ----------------------------------------------------------------------
# RGB GRID METADATA
# ----------------------------------------------------------------------

with rasterio.open(
    LOCAL_RGB_PATH
) as rgb_source:

    RGB_WIDTH = int(
        rgb_source.width
    )

    RGB_HEIGHT = int(
        rgb_source.height
    )

    RGB_TRANSFORM = (
        rgb_source.transform
    )

    RGB_CRS = rgb_source.crs

    RGB_RESOLUTION = abs(
        rgb_source.transform.a
    )

    RGB_PROFILE = (
        rgb_source.profile.copy()
    )


print(
    "RGB dimensions:",
    f"{RGB_WIDTH:,} x {RGB_HEIGHT:,}",
)

print(
    "RGB resolution:",
    RGB_RESOLUTION,
)


# ----------------------------------------------------------------------
# CREATE CHM STREAMING BLOCK BY BLOCK
# ----------------------------------------------------------------------

if (
    FORCE_REBUILD_LOCAL_INPUTS
    or not LOCAL_CHM_PATH.exists()
    or not LOCAL_DTM_COVERAGE_PATH.exists()
):

    if DTM_PATH is None:

        raise RuntimeError(
            "DTM_PATH is None.\n"
            "Set the partial DTM path in Block 03."
        )

    if not Path(
        DTM_PATH
    ).exists():

        raise FileNotFoundError(
            f"DTM not found:\n{DTM_PATH}"
        )

    print(
        "\nCreating CHM from DSM and partial DTM..."
    )

    chm_profile = {
        "driver": "GTiff",
        "height": RGB_HEIGHT,
        "width": RGB_WIDTH,
        "count": 1,
        "dtype": "float32",
        "crs": RGB_CRS,
        "transform": RGB_TRANSFORM,
        "nodata": CHM_NODATA,
        "compress": "LZW",
        "predictor": 3,
        "tiled": True,
        "blockxsize": OUTPUT_BLOCK_SIZE,
        "blockysize": OUTPUT_BLOCK_SIZE,
        "BIGTIFF": "YES",
    }

    coverage_profile = {
        "driver": "GTiff",
        "height": RGB_HEIGHT,
        "width": RGB_WIDTH,
        "count": 1,
        "dtype": "uint8",
        "crs": RGB_CRS,
        "transform": RGB_TRANSFORM,
        "nodata": MASK_NODATA,
        "compress": "LZW",
        "tiled": True,
        "blockxsize": OUTPUT_BLOCK_SIZE,
        "blockysize": OUTPUT_BLOCK_SIZE,
        "BIGTIFF": "YES",
    }

    with rasterio.open(
        DSM_PATH
    ) as dsm_source, rasterio.open(
        DTM_PATH
    ) as dtm_source, WarpedVRT(
        dsm_source,
        crs=RGB_CRS,
        transform=RGB_TRANSFORM,
        width=RGB_WIDTH,
        height=RGB_HEIGHT,
        resampling=Resampling.bilinear,
        nodata=CHM_NODATA,
    ) as dsm_vrt, WarpedVRT(
        dtm_source,
        crs=RGB_CRS,
        transform=RGB_TRANSFORM,
        width=RGB_WIDTH,
        height=RGB_HEIGHT,
        resampling=Resampling.bilinear,
        nodata=CHM_NODATA,
    ) as dtm_vrt, rasterio.open(
        LOCAL_CHM_PATH,
        "w",
        **chm_profile,
    ) as chm_destination, rasterio.open(
        LOCAL_DTM_COVERAGE_PATH,
        "w",
        **coverage_profile,
    ) as coverage_destination:

        total_valid_pixels = 0
        total_pixels = 0

        n_block_rows = int(
            np.ceil(
                RGB_HEIGHT
                / OUTPUT_BLOCK_SIZE
            )
        )

        n_block_cols = int(
            np.ceil(
                RGB_WIDTH
                / OUTPUT_BLOCK_SIZE
            )
        )

        total_blocks = (
            n_block_rows
            * n_block_cols
        )

        from tqdm.auto import tqdm

        progress = tqdm(
            total=total_blocks,
            desc="Streaming CHM creation",
        )

        for row_off in range(
            0,
            RGB_HEIGHT,
            OUTPUT_BLOCK_SIZE,
        ):

            block_height = min(
                OUTPUT_BLOCK_SIZE,
                RGB_HEIGHT - row_off,
            )

            for col_off in range(
                0,
                RGB_WIDTH,
                OUTPUT_BLOCK_SIZE,
            ):

                block_width = min(
                    OUTPUT_BLOCK_SIZE,
                    RGB_WIDTH - col_off,
                )

                window = Window(
                    col_off=col_off,
                    row_off=row_off,
                    width=block_width,
                    height=block_height,
                )

                dsm_block = dsm_vrt.read(
                    1,
                    window=window,
                    out_dtype="float32",
                )

                dtm_block = dtm_vrt.read(
                    1,
                    window=window,
                    out_dtype="float32",
                )

                valid_dsm = (
                    np.isfinite(
                        dsm_block
                    )
                    & (
                        dsm_block
                        != CHM_NODATA
                    )
                )

                valid_dtm = (
                    np.isfinite(
                        dtm_block
                    )
                    & (
                        dtm_block
                        != CHM_NODATA
                    )
                )

                valid = (
                    valid_dsm
                    & valid_dtm
                )

                chm_block = np.full(
                    (
                        block_height,
                        block_width,
                    ),
                    CHM_NODATA,
                    dtype=np.float32,
                )

                chm_block[
                    valid
                ] = (
                    dsm_block[
                        valid
                    ]
                    - dtm_block[
                        valid
                    ]
                )

                chm_block[
                    valid
                ] = np.clip(
                    chm_block[
                        valid
                    ],
                    0,
                    None,
                )

                coverage_block = np.zeros(
                    (
                        block_height,
                        block_width,
                    ),
                    dtype=np.uint8,
                )

                coverage_block[
                    valid
                ] = 1

                coverage_block[
                    ~valid_dsm
                ] = MASK_NODATA

                chm_destination.write(
                    chm_block,
                    1,
                    window=window,
                )

                coverage_destination.write(
                    coverage_block,
                    1,
                    window=window,
                )

                total_valid_pixels += int(
                    valid.sum()
                )

                total_pixels += int(
                    valid_dsm.sum()
                )

                progress.update(1)

        progress.close()

        coverage_fraction = (
            total_valid_pixels
            / max(
                total_pixels,
                1,
            )
        )

        chm_destination.update_tags(
            product="CHM_from_partial_DTM",
            formula="DSM - DTM",
            dtm_coverage_fraction=(
                coverage_fraction
            ),
        )

        coverage_destination.update_tags(
            class_0="no_valid_DTM",
            class_1="valid_DTM_and_CHM",
            class_255="no_valid_DSM",
        )

    print(
        "DTM/CHM coverage:",
        f"{coverage_fraction:.2%}",
    )

else:

    print(
        "Reusing existing partial-DTM CHM."
    )


# ----------------------------------------------------------------------
# ALIGN MUSANGA MASK BLOCK-WISE
# ----------------------------------------------------------------------

if (
    FORCE_REBUILD_LOCAL_INPUTS
    or not LOCAL_MUSANGA_MASK_PATH.exists()
):

    print(
        "\nAligning Musanga mask..."
    )

    mask_profile = {
        "driver": "GTiff",
        "height": RGB_HEIGHT,
        "width": RGB_WIDTH,
        "count": 1,
        "dtype": "uint8",
        "crs": RGB_CRS,
        "transform": RGB_TRANSFORM,
        "nodata": MASK_NODATA,
        "compress": "LZW",
        "tiled": True,
        "blockxsize": OUTPUT_BLOCK_SIZE,
        "blockysize": OUTPUT_BLOCK_SIZE,
        "BIGTIFF": "YES",
    }

    with rasterio.open(
        MUSANGA_MASK_PATH
    ) as source, WarpedVRT(
        source,
        crs=RGB_CRS,
        transform=RGB_TRANSFORM,
        width=RGB_WIDTH,
        height=RGB_HEIGHT,
        resampling=Resampling.nearest,
        nodata=MASK_NODATA,
    ) as mask_vrt, rasterio.open(
        LOCAL_MUSANGA_MASK_PATH,
        "w",
        **mask_profile,
    ) as destination:

        for row_off in range(
            0,
            RGB_HEIGHT,
            OUTPUT_BLOCK_SIZE,
        ):

            block_height = min(
                OUTPUT_BLOCK_SIZE,
                RGB_HEIGHT - row_off,
            )

            for col_off in range(
                0,
                RGB_WIDTH,
                OUTPUT_BLOCK_SIZE,
            ):

                block_width = min(
                    OUTPUT_BLOCK_SIZE,
                    RGB_WIDTH - col_off,
                )

                window = Window(
                    col_off=col_off,
                    row_off=row_off,
                    width=block_width,
                    height=block_height,
                )

                mask_block = mask_vrt.read(
                    1,
                    window=window,
                    out_dtype="uint8",
                )

                destination.write(
                    mask_block,
                    1,
                    window=window,
                )

else:

    print(
        "Reusing aligned Musanga mask."
    )


# ----------------------------------------------------------------------
# FINAL CHECK
# ----------------------------------------------------------------------

print("\nPrepared local inputs:")

for name, path in {
    "Tiled RGB": LOCAL_RGB_PATH,
    "Partial-DTM CHM": LOCAL_CHM_PATH,
    "DTM coverage": LOCAL_DTM_COVERAGE_PATH,
    "Musanga mask": LOCAL_MUSANGA_MASK_PATH,
}.items():

    print(
        f"{name}:",
        path,
        "->",
        "FOUND"
        if path.exists()
        else "MISSING",
    )


BLOCK 05 — MEMORY-SAFE TILED RGB AND PARTIAL-DTM CHM
Reusing local tiled RGB.
RGB dimensions: 44,359 x 23,217
RGB resolution: 0.07027
Reusing existing partial-DTM CHM.
Reusing aligned Musanga mask.

Prepared local inputs:
Tiled RGB: /content/mask_rcnn_sources/rgb_tiled_512.tif -> FOUND
Partial-DTM CHM: /content/mask_rcnn_sources/chm_partial_dtm_aligned.tif -> FOUND
DTM coverage: /content/mask_rcnn_sources/dtm_coverage_aligned.tif -> FOUND
Musanga mask: /content/mask_rcnn_sources/musanga_mask_aligned.tif -> FOUND


## BLOCK 06 — Load and prepare non-Musanga crown labels

In [6]:

# ======================================================================================
# BLOCK 06 — LOAD AND PREPARE NON-MUSANGA CROWN LABELS
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 06 — LOAD AND PREPARE NON-MUSANGA CROWN LABELS")
print("=" * 100)

crowns = gpd.read_file(
    REFERENCE_CROWNS_PATH
)

if crowns.crs != RGB_CRS:
    crowns = crowns.to_crs(
        RGB_CRS
    )

class_candidates = [
    "Clss_ID",
    "class_id",
    "reference_class",
    "class",
]

CLASS_COLUMN = next(
    (
        column
        for column in class_candidates
        if column in crowns.columns
    ),
    None,
)

if CLASS_COLUMN is None:
    raise ValueError(
        "Could not identify the class column. "
        f"Available columns: {crowns.columns.tolist()}"
    )

crowns = crowns[
    crowns[CLASS_COLUMN] != 1
].copy()

crowns = crowns[
    crowns.geometry.notna()
    & ~crowns.geometry.is_empty
].copy()

crowns["geometry"] = crowns.geometry.buffer(0)
crowns["area_m2"] = crowns.geometry.area

crowns = crowns[
    crowns["area_m2"].between(
        MIN_CROWN_AREA_M2,
        MAX_CROWN_AREA_M2,
    )
].copy()

crowns = crowns.reset_index(drop=True)
crowns["crown_id"] = np.arange(
    1,
    len(crowns) + 1,
)

print("Non-Musanga crown polygons:", len(crowns))
print(
    "Median polygon area:",
    f"{crowns['area_m2'].median():.2f} m²",
)



BLOCK 06 — LOAD AND PREPARE NON-MUSANGA CROWN LABELS
Non-Musanga crown polygons: 3344
Median polygon area: 23.75 m²


## BLOCK 07 — Optional CHM-guided expansion of conservative crown cores

In [7]:

# ======================================================================================
# BLOCK 07 — OPTIONAL CHM-GUIDED EXPANSION OF CONSERVATIVE CROWN CORES
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 07 — OPTIONAL CHM-GUIDED EXPANSION OF CONSERVATIVE CROWN CORES")
print("=" * 100)

EXPANDED_LABEL_PATH = (
    MODEL_DIR
    / "non_musanga_training_crowns_expanded.gpkg"
)

if LABEL_MODE == "ORIGINAL":

    training_crowns = crowns.copy()
    training_crowns["label_source"] = "original"

elif EXPANDED_LABEL_PATH.exists():

    print("Reusing expanded crown labels.")
    training_crowns = gpd.read_file(
        EXPANDED_LABEL_PATH
    )

else:

    print("Creating CHM-guided expanded labels...")

    expanded_records = []

    with rasterio.open(
        LOCAL_CHM_PATH
    ) as chm_src, rasterio.open(
        LOCAL_RGB_PATH
    ) as rgb_src:

        for record in tqdm(
            crowns.itertuples(),
            total=len(crowns),
            desc="Expanding crown cores",
        ):

            core = record.geometry

            search_geometry = core.buffer(
                MAX_EXPANSION_DISTANCE_M
            )

            window = rasterio.windows.from_bounds(
                *search_geometry.bounds,
                transform=RGB_TRANSFORM,
            ).round_offsets().round_lengths()

            try:
                window = window.intersection(
                    Window(
                        0,
                        0,
                        RGB_WIDTH,
                        RGB_HEIGHT,
                    )
                )
            except Exception:
                continue

            height = int(window.height)
            width = int(window.width)

            if height < 3 or width < 3:
                continue

            local_transform = chm_src.window_transform(
                window
            )

            chm_patch = chm_src.read(
                1,
                window=window,
            )

            rgb_patch = np.moveaxis(
                rgb_src.read(
                    [1, 2, 3],
                    window=window,
                ),
                0,
                -1,
            ).astype(np.float32) / 255.0

            core_mask = rasterize(
                [(core, 1)],
                out_shape=(height, width),
                transform=local_transform,
                fill=0,
                dtype=np.uint8,
            ).astype(bool)

            search_mask = rasterize(
                [(search_geometry, 1)],
                out_shape=(height, width),
                transform=local_transform,
                fill=0,
                dtype=np.uint8,
            ).astype(bool)

            valid_height = (
                (chm_patch != -9999.0)
                & (chm_patch >= MIN_TREE_HEIGHT_M)
            )

            if not core_mask.any():
                continue

            # Marker 1 = target crown core.
            # Marker 2 = outside the allowable search region or low vegetation.
            markers = np.zeros(
                (height, width),
                dtype=np.int32,
            )
            markers[core_mask] = 1
            markers[
                ~search_mask
                | ~valid_height
            ] = 2

            # Combine CHM slope and RGB intensity gradient.
            chm_fill = np.where(
                chm_patch == -9999.0,
                0,
                chm_patch,
            )
            chm_gradient = ndi.gaussian_gradient_magnitude(
                chm_fill,
                sigma=max(
                    0.5,
                    0.5 / RGB_RESOLUTION,
                ),
            )

            gray = (
                0.2989 * rgb_patch[..., 0]
                + 0.5870 * rgb_patch[..., 1]
                + 0.1140 * rgb_patch[..., 2]
            )
            rgb_gradient = ndi.gaussian_gradient_magnitude(
                gray,
                sigma=1.0,
            )

            surface = (
                0.70
                * chm_gradient
                / max(
                    np.nanpercentile(
                        chm_gradient,
                        99,
                    ),
                    1e-6,
                )
                + 0.30
                * rgb_gradient
                / max(
                    np.nanpercentile(
                        rgb_gradient,
                        99,
                    ),
                    1e-6,
                )
            )

            labels = segmentation.watershed(
                surface,
                markers=markers,
                mask=search_mask,
            )

            expanded_mask = labels == 1

            polygon_candidates = []

            for geom_mapping, value in shapes(
                expanded_mask.astype(np.uint8),
                mask=expanded_mask,
                transform=local_transform,
            ):
                if int(value) != 1:
                    continue
                polygon_candidates.append(
                    shape(geom_mapping)
                )

            if not polygon_candidates:
                expanded_geometry = core
                label_source = "original_fallback"
            else:
                expanded_geometry = max(
                    polygon_candidates,
                    key=lambda geometry: geometry.area,
                ).buffer(0)

                intersection = (
                    expanded_geometry.intersection(core).area
                )
                union = (
                    expanded_geometry.union(core).area
                )
                iou = (
                    intersection / union
                    if union > 0
                    else 0
                )

                if (
                    iou < MIN_EXPANDED_IOU_WITH_CORE
                    or expanded_geometry.area
                    > MAX_CROWN_AREA_M2
                ):
                    expanded_geometry = core
                    label_source = "original_fallback"
                else:
                    label_source = "chm_expanded"

            expanded_records.append(
                {
                    "crown_id": record.crown_id,
                    "label_source": label_source,
                    "original_area_m2": core.area,
                    "area_m2": expanded_geometry.area,
                    "geometry": expanded_geometry,
                }
            )

    training_crowns = gpd.GeoDataFrame(
        expanded_records,
        geometry="geometry",
        crs=RGB_CRS,
    )

    training_crowns.to_file(
        EXPANDED_LABEL_PATH,
        layer="crowns",
        driver="GPKG",
    )

print("Training crowns:", len(training_crowns))
print(
    training_crowns[
        "label_source"
    ].value_counts(
        dropna=False
    )
)
print(
    "Median training-label area:",
    f"{training_crowns.geometry.area.median():.2f} m²",
)



BLOCK 07 — OPTIONAL CHM-GUIDED EXPANSION OF CONSERVATIVE CROWN CORES
Reusing expanded crown labels.
Training crowns: 998
label_source
original_fallback    742
chm_expanded         256
Name: count, dtype: int64
Median training-label area: 35.99 m²


## BLOCK 08 — Spatial train/validation/test split

In [8]:

# ======================================================================================
# BLOCK 08 — SPATIAL TRAIN / VALIDATION / TEST SPLIT
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 08 — SPATIAL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 100)

centroids = training_crowns.geometry.centroid

training_crowns["block_x"] = np.floor(
    centroids.x
    / SPATIAL_BLOCK_SIZE_M
).astype(int)

training_crowns["block_y"] = np.floor(
    centroids.y
    / SPATIAL_BLOCK_SIZE_M
).astype(int)

training_crowns["spatial_block"] = (
    training_crowns["block_x"].astype(str)
    + "_"
    + training_crowns["block_y"].astype(str)
)

blocks = (
    training_crowns[
        "spatial_block"
    ]
    .drop_duplicates()
    .sample(
        frac=1.0,
        random_state=RANDOM_SEED,
    )
    .tolist()
)

n_blocks = len(blocks)
n_train = max(
    1,
    int(round(
        TRAIN_FRACTION
        * n_blocks
    )),
)
n_validation = max(
    1,
    int(round(
        VALIDATION_FRACTION
        * n_blocks
    )),
)

train_blocks = set(
    blocks[:n_train]
)
validation_blocks = set(
    blocks[
        n_train:
        n_train + n_validation
    ]
)
test_blocks = set(
    blocks[
        n_train + n_validation:
    ]
)

def assign_split(block):
    if block in train_blocks:
        return "train"
    if block in validation_blocks:
        return "validation"
    return "test"

training_crowns["split"] = (
    training_crowns[
        "spatial_block"
    ].map(assign_split)
)

split_summary = (
    training_crowns[
        "split"
    ]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="n_crowns")
)

display(split_summary)

SPLIT_CROWNS_PATH = (
    MODEL_DIR
    / "mask_rcnn_training_crowns_with_split.gpkg"
)

training_crowns.to_file(
    SPLIT_CROWNS_PATH,
    layer="crowns",
    driver="GPKG",
)

print("Saved:", SPLIT_CROWNS_PATH)



BLOCK 08 — SPATIAL TRAIN / VALIDATION / TEST SPLIT


,split,n_crowns
0,train,662
1,test,198
2,validation,138


Saved: /content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga/mask_rcnn_training_crowns_with_split.gpkg


## BLOCK 09 — Build crown-centred patch index

In [9]:

# ======================================================================================
# BLOCK 09 — BUILD CROWN-CENTRED PATCH INDEX
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 09 — BUILD CROWN-CENTRED PATCH INDEX")
print("=" * 100)

PATCH_INDEX_PATH = (
    CACHE_DIR
    / (
        "mask_rcnn_patch_index_"
        f"{LABEL_MODE.lower()}_"
        f"{PATCH_SIZE}.csv"
    )
)

if PATCH_INDEX_PATH.exists():

    patch_index = pd.read_csv(
        PATCH_INDEX_PATH
    )
    print("Reusing patch index.")

else:

    records = []

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        split_crowns = training_crowns[
            training_crowns["split"]
            == split_name
        ]

        for crown in split_crowns.itertuples():

            centroid = crown.geometry.centroid

            col, row = (
                ~RGB_TRANSFORM
                * (
                    centroid.x,
                    centroid.y,
                )
            )

            col_off = int(
                round(
                    col - PATCH_SIZE / 2
                )
            )
            row_off = int(
                round(
                    row - PATCH_SIZE / 2
                )
            )

            col_off = max(
                0,
                min(
                    col_off,
                    RGB_WIDTH - PATCH_SIZE,
                ),
            )
            row_off = max(
                0,
                min(
                    row_off,
                    RGB_HEIGHT - PATCH_SIZE,
                ),
            )

            records.append(
                {
                    "patch_id": (
                        f"{split_name}_"
                        f"r{row_off}_"
                        f"c{col_off}"
                    ),
                    "split": split_name,
                    "row_off": row_off,
                    "col_off": col_off,
                    "height": PATCH_SIZE,
                    "width": PATCH_SIZE,
                }
            )

    patch_index = (
        pd.DataFrame(records)
        .drop_duplicates(
            subset=[
                "split",
                "row_off",
                "col_off",
            ]
        )
        .reset_index(drop=True)
    )

    patch_index.to_csv(
        PATCH_INDEX_PATH,
        index=False,
    )

def limit_split(
    table,
    split_name,
    maximum,
):
    subset = table[
        table["split"] == split_name
    ].copy()

    if (
        maximum is not None
        and len(subset) > maximum
    ):
        subset = subset.sample(
            n=maximum,
            random_state=RANDOM_SEED,
        )

    return subset.reset_index(drop=True)

train_patch_index = limit_split(
    patch_index,
    "train",
    MAX_TRAIN_PATCHES,
)
validation_patch_index = limit_split(
    patch_index,
    "validation",
    MAX_VALIDATION_PATCHES,
)
test_patch_index = limit_split(
    patch_index,
    "test",
    MAX_TEST_PATCHES,
)

print("Train patches:", len(train_patch_index))
print("Validation patches:", len(validation_patch_index))
print("Test patches:", len(test_patch_index))



BLOCK 09 — BUILD CROWN-CENTRED PATCH INDEX
Reusing patch index.
Train patches: 662
Validation patches: 138
Test patches: 198


## BLOCK 10 — Dataset and DataLoaders

In [10]:

# ======================================================================================
# BLOCK 10 — DATASET AND DATALOADERS
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 10 — DATASET AND DATALOADERS")
print("=" * 100)

# Robust CHM normalization range from sampled crown heights.
with rasterio.open(
    LOCAL_CHM_PATH
) as chm_src:

    sample_values = []

    for row in train_patch_index.head(
        min(
            100,
            len(train_patch_index),
        )
    ).itertuples():

        patch = chm_src.read(
            1,
            window=Window(
                row.col_off,
                row.row_off,
                row.width,
                row.height,
            ),
        )

        values = patch[
            (patch != -9999.0)
            & np.isfinite(patch)
        ]

        if len(values):
            sample_values.append(
                values[
                    ::max(
                        1,
                        len(values) // 5000,
                    )
                ]
            )

    sample_values = np.concatenate(
        sample_values
    )

CHM_LOW = float(
    np.percentile(
        sample_values,
        1,
    )
)
CHM_HIGH = float(
    np.percentile(
        sample_values,
        99,
    )
)

print("CHM normalization:", CHM_LOW, "to", CHM_HIGH)


class CrownInstanceDataset(Dataset):

    def __init__(
        self,
        patch_table,
        crowns_gdf,
        augment=False,
    ):
        self.patch_table = (
            patch_table.reset_index(
                drop=True
            )
        )
        self.crowns = crowns_gdf
        self.augment = augment

    def __len__(self):
        return len(
            self.patch_table
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.patch_table.iloc[
            index
        ]

        window = Window(
            int(row.col_off),
            int(row.row_off),
            int(row.width),
            int(row.height),
        )

        with rasterio.open(
            LOCAL_RGB_PATH
        ) as rgb_src:
            rgb = rgb_src.read(
                [1, 2, 3],
                window=window,
                boundless=True,
                fill_value=0,
            ).astype(np.float32) / 255.0

            transform = (
                rgb_src.window_transform(
                    window
                )
            )

            bounds = rasterio.windows.bounds(
                window,
                rgb_src.transform,
            )

        with rasterio.open(
            LOCAL_CHM_PATH
        ) as chm_src:
            chm = chm_src.read(
                1,
                window=window,
                boundless=True,
                fill_value=-9999.0,
            ).astype(np.float32)

        chm_valid = (
            chm != -9999.0
        )

        chm_normalized = np.zeros_like(
            chm,
            dtype=np.float32,
        )

        chm_normalized[
            chm_valid
        ] = np.clip(
            (
                chm[chm_valid]
                - CHM_LOW
            )
            / max(
                CHM_HIGH
                - CHM_LOW,
                1e-6,
            ),
            0,
            1,
        )

        image = np.concatenate(
            [
                rgb,
                chm_normalized[
                    None,
                    ...,
                ],
            ],
            axis=0,
        )

        patch_geometry = box(
            *bounds
        )

        candidate_indices = list(
            self.crowns.sindex.intersection(
                patch_geometry.bounds
            )
        )

        patch_crowns = self.crowns.iloc[
            candidate_indices
        ].copy()

        patch_crowns = patch_crowns[
            patch_crowns.intersects(
                patch_geometry
            )
        ].copy()

        masks = []
        boxes = []
        areas = []

        for crown_geometry in patch_crowns.geometry:

            clipped = (
                crown_geometry.intersection(
                    patch_geometry
                )
            )

            if (
                clipped.is_empty
                or clipped.area
                < MIN_CROWN_AREA_M2
            ):
                continue

            mask = rasterize(
                [(clipped, 1)],
                out_shape=(
                    PATCH_SIZE,
                    PATCH_SIZE,
                ),
                transform=transform,
                fill=0,
                dtype=np.uint8,
            )

            if mask.sum() < 4:
                continue

            rows, cols = np.where(
                mask > 0
            )

            xmin = float(
                cols.min()
            )
            xmax = float(
                cols.max() + 1
            )
            ymin = float(
                rows.min()
            )
            ymax = float(
                rows.max() + 1
            )

            if (
                xmax <= xmin
                or ymax <= ymin
            ):
                continue

            masks.append(mask)
            boxes.append(
                [
                    xmin,
                    ymin,
                    xmax,
                    ymax,
                ]
            )
            areas.append(
                float(mask.sum())
            )

        if masks:

            masks = torch.as_tensor(
                np.stack(masks),
                dtype=torch.uint8,
            )

            boxes = torch.as_tensor(
                boxes,
                dtype=torch.float32,
            )

            labels = torch.ones(
                (len(masks),),
                dtype=torch.int64,
            )

            areas = torch.as_tensor(
                areas,
                dtype=torch.float32,
            )

            iscrowd = torch.zeros(
                (len(masks),),
                dtype=torch.int64,
            )

        else:

            masks = torch.zeros(
                (
                    0,
                    PATCH_SIZE,
                    PATCH_SIZE,
                ),
                dtype=torch.uint8,
            )

            boxes = torch.zeros(
                (0, 4),
                dtype=torch.float32,
            )

            labels = torch.zeros(
                (0,),
                dtype=torch.int64,
            )

            areas = torch.zeros(
                (0,),
                dtype=torch.float32,
            )

            iscrowd = torch.zeros(
                (0,),
                dtype=torch.int64,
            )

        image = torch.as_tensor(
            image,
            dtype=torch.float32,
        )

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor(
                [index],
                dtype=torch.int64,
            ),
            "area": areas,
            "iscrowd": iscrowd,
        }

        if self.augment:

            if random.random() < 0.5:
                image = torch.flip(
                    image,
                    dims=[2],
                )
                target["masks"] = torch.flip(
                    target["masks"],
                    dims=[2],
                )

                if len(target["boxes"]):
                    old_xmin = (
                        target["boxes"][:, 0]
                        .clone()
                    )
                    old_xmax = (
                        target["boxes"][:, 2]
                        .clone()
                    )
                    target["boxes"][:, 0] = (
                        PATCH_SIZE - old_xmax
                    )
                    target["boxes"][:, 2] = (
                        PATCH_SIZE - old_xmin
                    )

            if random.random() < 0.5:
                image = torch.flip(
                    image,
                    dims=[1],
                )
                target["masks"] = torch.flip(
                    target["masks"],
                    dims=[1],
                )

                if len(target["boxes"]):
                    old_ymin = (
                        target["boxes"][:, 1]
                        .clone()
                    )
                    old_ymax = (
                        target["boxes"][:, 3]
                        .clone()
                    )
                    target["boxes"][:, 1] = (
                        PATCH_SIZE - old_ymax
                    )
                    target["boxes"][:, 3] = (
                        PATCH_SIZE - old_ymin
                    )

        return image, target


def collate_fn(batch):
    return tuple(
        zip(*batch)
    )


train_dataset = CrownInstanceDataset(
    train_patch_index,
    training_crowns[
        training_crowns["split"]
        == "train"
    ],
    augment=True,
)

validation_dataset = CrownInstanceDataset(
    validation_patch_index,
    training_crowns[
        training_crowns["split"]
        == "validation"
    ],
    augment=False,
)

test_dataset = CrownInstanceDataset(
    test_patch_index,
    training_crowns[
        training_crowns["split"]
        == "test"
    ],
    augment=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

print("DataLoaders ready.")



BLOCK 10 — DATASET AND DATALOADERS
CHM normalization: 12.900605201721191 to 50.87058639526367
DataLoaders ready.


## BLOCK 11 — Build four-channel Mask R-CNN

In [11]:
# ======================================================================================
# BLOCK 11 — BUILD FOUR-CHANNEL MASK R-CNN
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 11 — BUILD FOUR-CHANNEL MASK R-CNN")
print("=" * 100)

import torch

from torchvision.models.detection import (
    MaskRCNN_ResNet50_FPN_Weights,
    maskrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor,
)
from torchvision.models.detection.mask_rcnn import (
    MaskRCNNPredictor,
)


def build_mask_rcnn_4channel():
    """
    Build a Mask R-CNN model for one foreground class:

        0 = background
        1 = non-Musanga crown

    Input channels:

        1 = red
        2 = green
        3 = blue
        4 = normalized CHM
    """

    # ------------------------------------------------------------------
    # LOAD PRETRAINED MASK R-CNN
    # ------------------------------------------------------------------

    model = maskrcnn_resnet50_fpn(
        weights=(
            MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        ),
    )

    # ------------------------------------------------------------------
    # MODIFY THE RESNET INPUT LAYER FROM 3 TO 4 CHANNELS
    # ------------------------------------------------------------------

    old_conv = model.backbone.body.conv1

    new_conv = torch.nn.Conv2d(
        in_channels=4,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        dilation=old_conv.dilation,
        groups=old_conv.groups,
        bias=False,
        padding_mode=old_conv.padding_mode,
    )

    with torch.no_grad():

        # Preserve the pretrained RGB weights.
        new_conv.weight[:, :3] = (
            old_conv.weight
        )

        # Initialize the CHM channel using the mean RGB kernel.
        new_conv.weight[:, 3:4] = (
            old_conv.weight.mean(
                dim=1,
                keepdim=True,
            )
        )

    model.backbone.body.conv1 = new_conv

    # ------------------------------------------------------------------
    # REPLACE BOX CLASSIFIER
    #
    # Classes:
    #   0 = background
    #   1 = non-Musanga crown
    # ------------------------------------------------------------------

    num_classes = 2

    box_in_features = (
        model.roi_heads
        .box_predictor
        .cls_score
        .in_features
    )

    model.roi_heads.box_predictor = (
        FastRCNNPredictor(
            box_in_features,
            num_classes,
        )
    )

    # ------------------------------------------------------------------
    # REPLACE MASK PREDICTOR
    # ------------------------------------------------------------------

    mask_in_features = (
        model.roi_heads
        .mask_predictor
        .conv5_mask
        .in_channels
    )

    mask_hidden_channels = 256

    model.roi_heads.mask_predictor = (
        MaskRCNNPredictor(
            mask_in_features,
            mask_hidden_channels,
            num_classes,
        )
    )

    # ------------------------------------------------------------------
    # FOUR-CHANNEL NORMALIZATION
    #
    # RGB uses ImageNet statistics.
    # CHM is normalized to approximately 0–1 in the Dataset.
    # ------------------------------------------------------------------

    model.transform.image_mean = [
        0.485,
        0.456,
        0.406,
        0.50,
    ]

    model.transform.image_std = [
        0.229,
        0.224,
        0.225,
        0.25,
    ]

    # ------------------------------------------------------------------
    # KEEP 512 × 512 PATCHES AT THEIR ORIGINAL SIZE
    #
    # TorchVision expects:
    #   min_size = sequence of candidate sizes
    #   max_size = one integer
    # ------------------------------------------------------------------

    model.transform.min_size = [
        int(PATCH_SIZE)
    ]

    model.transform.max_size = int(
        PATCH_SIZE
    )

    return model


# ----------------------------------------------------------------------
# CREATE MODEL
# ----------------------------------------------------------------------

model = build_mask_rcnn_4channel()

model = model.to(
    DEVICE
)


# ----------------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------------

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Device:", DEVICE)

print(
    "Input channels:",
    model.backbone.body.conv1.in_channels,
)

print(
    "Patch resize setting:",
    model.transform.min_size,
    model.transform.max_size,
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}",
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

print(
    "Mask R-CNN created successfully."
)


BLOCK 11 — BUILD FOUR-CHANNEL MASK R-CNN
Device: cuda
Input channels: 4
Patch resize setting: [512] 512
Trainable parameters: 43,712,539
Total parameters: 43,925,531
Mask R-CNN created successfully.


## BLOCK 12 — Training loop

In [54]:

# ======================================================================================
# BLOCK 12 — TRAINING LOOP
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 12 — TRAINING LOOP")
print("=" * 100)

CHECKPOINT_PATH = (
    MODEL_DIR
    / (
        "best_mask_rcnn_rgb_chm_"
        f"{LABEL_MODE.lower()}_"
        f"{'quick' if QUICK_TEST else 'full'}.pt"
    )
)

optimizer = torch.optim.SGD(
    [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=max(
        3,
        EPOCHS // 3,
    ),
    gamma=0.3,
)


def train_one_epoch(
    model,
    loader,
    optimizer,
):
    model.train()

    total_loss = 0.0
    n_batches = 0

    progress = tqdm(
        loader,
        desc="Training",
    )

    for images, targets in progress:

        images = [
            image.to(
                DEVICE,
                non_blocking=True,
            )
            for image in images
        ]

        targets = [
            {
                key: value.to(
                    DEVICE,
                    non_blocking=True,
                )
                for key, value in target.items()
            }
            for target in targets
        ]

        loss_dict = model(
            images,
            targets,
        )

        loss = sum(
            loss_value
            for loss_value
            in loss_dict.values()
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        total_loss += float(
            loss.item()
        )
        n_batches += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        total_loss
        / max(
            n_batches,
            1,
        )
    )


def validation_loss(
    model,
    loader,
):
    # Detection models return losses only in train mode.
    model.train()

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():

        for images, targets in tqdm(
            loader,
            desc="Validation loss",
        ):

            images = [
                image.to(
                    DEVICE,
                    non_blocking=True,
                )
                for image in images
            ]

            targets = [
                {
                    key: value.to(
                        DEVICE,
                        non_blocking=True,
                    )
                    for key, value in target.items()
                }
                for target in targets
            ]

            loss_dict = model(
                images,
                targets,
            )

            loss = sum(
                value
                for value
                in loss_dict.values()
            )

            total_loss += float(
                loss.item()
            )
            n_batches += 1

    model.eval()

    return (
        total_loss
        / max(
            n_batches,
            1,
        )
    )


history = []
best_validation_loss = np.inf

if RUN_TRAINING:

    for epoch in range(
        1,
        EPOCHS + 1,
    ):

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
        )

        val_loss = validation_loss(
            model,
            validation_loader,
        )

        scheduler.step()

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": val_loss,
                "learning_rate": (
                    optimizer.param_groups[
                        0
                    ]["lr"]
                ),
            }
        )

        print(
            f"Epoch {epoch:03d} | "
            f"train loss={train_loss:.4f} | "
            f"validation loss={val_loss:.4f}"
        )

        if val_loss < best_validation_loss:

            best_validation_loss = (
                val_loss
            )

            torch.save(
                {
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),
                    "epoch": epoch,
                    "validation_loss": (
                        val_loss
                    ),
                    "label_mode": LABEL_MODE,
                    "chm_low": CHM_LOW,
                    "chm_high": CHM_HIGH,
                },
                CHECKPOINT_PATH,
            )

            print(
                "Saved best checkpoint:",
                CHECKPOINT_PATH,
            )

else:

    print(
        "Training skipped. "
        "The saved checkpoint will be loaded in Block 13."
    )

HISTORY_PATH = (
    MODEL_DIR
    / (
        "training_history_"
        f"{LABEL_MODE.lower()}_"
        f"{'quick' if QUICK_TEST else 'full'}.csv"
    )
)

pd.DataFrame(history).to_csv(
    HISTORY_PATH,
    index=False,
)



BLOCK 12 — TRAINING LOOP


Training:   0%|          | 0/166 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 001 | train loss=1.4532 | validation loss=1.2787
Saved best checkpoint: /content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga/best_mask_rcnn_rgb_chm_chm_expanded_full.pt


Training:   0%|          | 0/166 [00:00<?, ?it/s]

...

Training:   0%|          | 0/166 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 007 | train loss=0.7585 | validation loss=1.5421


Training:   0%|          | 0/166 [00:00<?, ?it/s]

KeyboardInterrupt: 

## BLOCK 13 — Reload best checkpoint

In [13]:
# ======================================================================================
# BLOCK 13 — RELOAD BEST CHECKPOINT WITHOUT RUNNING TRAINING
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 13 — RELOAD BEST CHECKPOINT WITHOUT RUNNING TRAINING")
print("=" * 100)

from pathlib import Path
import torch


# ----------------------------------------------------------------------
# RECONSTRUCT CHECKPOINT PATH
# ----------------------------------------------------------------------

CHECKPOINT_PATH = (
    MODEL_DIR
    / (
        "best_mask_rcnn_rgb_chm_"
        f"{LABEL_MODE.lower()}_"
        f"{'quick' if QUICK_TEST else 'full'}.pt"
    )
)

print("Expected checkpoint:")
print(CHECKPOINT_PATH)


# ----------------------------------------------------------------------
# VALIDATE CHECKPOINT
# ----------------------------------------------------------------------

if not CHECKPOINT_PATH.exists():

    available_checkpoints = sorted(
        MODEL_DIR.glob(
            "best_mask_rcnn_rgb_chm_*.pt"
        )
    )

    print("\nAvailable checkpoints:")

    for path in available_checkpoints:
        print(" -", path.name)

    raise FileNotFoundError(
        "The expected checkpoint was not found.\n"
        "Check QUICK_TEST and LABEL_MODE in Block 03."
    )


# ----------------------------------------------------------------------
# LOAD CHECKPOINT
# ----------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)


# ----------------------------------------------------------------------
# LOAD MODEL WEIGHTS
# ----------------------------------------------------------------------

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model = model.to(
    DEVICE
)

model.eval()


# ----------------------------------------------------------------------
# RESTORE SAVED NORMALIZATION VALUES WHEN PRESENT
# ----------------------------------------------------------------------

if "chm_low" in checkpoint:

    CHM_LOW = float(
        checkpoint[
            "chm_low"
        ]
    )

if "chm_high" in checkpoint:

    CHM_HIGH = float(
        checkpoint[
            "chm_high"
        ]
    )


# ----------------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------------

print("\nCheckpoint loaded successfully.")

print(
    "Saved epoch:",
    checkpoint.get(
        "epoch",
        "unknown",
    ),
)

print(
    "Saved validation loss:",
    checkpoint.get(
        "validation_loss",
        "unknown",
    ),
)

print(
    "Saved label mode:",
    checkpoint.get(
        "label_mode",
        "unknown",
    ),
)

print(
    "CHM normalization:",
    CHM_LOW,
    "to",
    CHM_HIGH,
)

print(
    "State-dict result:",
    load_result,
)


BLOCK 13 — RELOAD BEST CHECKPOINT WITHOUT RUNNING TRAINING
Expected checkpoint:
/content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga/best_mask_rcnn_rgb_chm_chm_expanded_full.pt

Checkpoint loaded successfully.
Saved epoch: 1
Saved validation loss: 1.2786945052768872
Saved label mode: CHM_EXPANDED
CHM normalization: 12.900605201721191 to 50.87058639526367
State-dict result: <All keys matched successfully>


## BLOCK 14 — Instance-level test evaluation

In [14]:
# ======================================================================================
# BLOCK 14 — MEMORY-SAFE VALIDATION THRESHOLD SWEEP AND TEST EVALUATION
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 14 — MEMORY-SAFE VALIDATION THRESHOLD SWEEP AND TEST EVALUATION")
print("=" * 100)

import gc
import json

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm


# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------

SCORE_THRESHOLDS = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]

MASK_THRESHOLD = 0.50

IOU_THRESHOLDS = [
    0.25,
    0.50,
]

# Choose the score threshold from validation performance using this IoU.
SELECTION_IOU_THRESHOLD = 0.50

# Avoid retaining very weak detections that will never be evaluated.
MINIMUM_SCORE_TO_RETAIN = min(
    SCORE_THRESHOLDS
)


# ----------------------------------------------------------------------
# INITIALIZE METRIC COUNTERS
# ----------------------------------------------------------------------

def initialize_counters(
    score_thresholds,
    iou_thresholds,
):

    counters = {}

    for score_threshold in score_thresholds:

        for iou_threshold in iou_thresholds:

            key = (
                float(score_threshold),
                float(iou_threshold),
            )

            counters[key] = {
                "true_positive_crowns": 0,
                "false_positive_crowns": 0,
                "false_negative_crowns": 0,
                "n_reference_crowns": 0,
                "n_predicted_crowns": 0,
                "matched_iou_sum": 0.0,
                "matched_iou_count": 0,
                "matched_ious": [],
            }

    return counters


# ----------------------------------------------------------------------
# COMPUTE PAIRWISE MASK IoU FOR ONE PATCH
# ----------------------------------------------------------------------

def compute_pairwise_mask_iou(
    predicted_masks,
    reference_masks,
):
    """
    Return an array of shape:

        n_predictions × n_references

    Only one patch is processed at a time.
    """

    n_predictions = len(
        predicted_masks
    )

    n_references = len(
        reference_masks
    )

    iou_matrix = np.zeros(
        (
            n_predictions,
            n_references,
        ),
        dtype=np.float32,
    )

    if (
        n_predictions == 0
        or n_references == 0
    ):
        return iou_matrix

    predicted_flat = (
        predicted_masks
        .reshape(
            n_predictions,
            -1,
        )
        .astype(
            np.uint8,
            copy=False,
        )
    )

    reference_flat = (
        reference_masks
        .reshape(
            n_references,
            -1,
        )
        .astype(
            np.uint8,
            copy=False,
        )
    )

    predicted_areas = (
        predicted_flat.sum(
            axis=1,
            dtype=np.int64,
        )
    )

    reference_areas = (
        reference_flat.sum(
            axis=1,
            dtype=np.int64,
        )
    )

    # Matrix multiplication gives all pairwise intersection counts.
    intersections = (
        predicted_flat.astype(
            np.float32
        )
        @ reference_flat.T.astype(
            np.float32
        )
    )

    unions = (
        predicted_areas[:, None]
        + reference_areas[None, :]
        - intersections
    )

    valid_union = (
        unions > 0
    )

    iou_matrix[
        valid_union
    ] = (
        intersections[
            valid_union
        ]
        / unions[
            valid_union
        ]
    )

    return iou_matrix


# ----------------------------------------------------------------------
# GREEDY ONE-TO-ONE MATCHING
# ----------------------------------------------------------------------

def greedy_match_from_iou_matrix(
    iou_matrix,
    iou_threshold,
):

    if iou_matrix.size == 0:

        return (
            0,
            [],
        )

    prediction_indices, reference_indices = (
        np.where(
            iou_matrix
            >= iou_threshold
        )
    )

    if len(
        prediction_indices
    ) == 0:

        return (
            0,
            [],
        )

    candidate_ious = iou_matrix[
        prediction_indices,
        reference_indices,
    ]

    order = np.argsort(
        candidate_ious
    )[::-1]

    matched_predictions = set()
    matched_references = set()
    matched_ious = []

    for candidate_index in order:

        predicted_index = int(
            prediction_indices[
                candidate_index
            ]
        )

        reference_index = int(
            reference_indices[
                candidate_index
            ]
        )

        if (
            predicted_index
            in matched_predictions
        ):
            continue

        if (
            reference_index
            in matched_references
        ):
            continue

        matched_predictions.add(
            predicted_index
        )

        matched_references.add(
            reference_index
        )

        matched_ious.append(
            float(
                candidate_ious[
                    candidate_index
                ]
            )
        )

    return (
        len(
            matched_ious
        ),
        matched_ious,
    )


# ----------------------------------------------------------------------
# STREAM ONE DATASET THROUGH THE MODEL
# ----------------------------------------------------------------------

def stream_evaluate_loader(
    model,
    loader,
    score_thresholds,
    iou_thresholds,
    mask_threshold,
    split_name,
):

    counters = initialize_counters(
        score_thresholds,
        iou_thresholds,
    )

    model.eval()

    with torch.inference_mode():

        for images, targets in tqdm(
            loader,
            desc=(
                f"{split_name}: "
                "streaming inference"
            ),
        ):

            # The loaders currently use batch size 1, but this works for
            # any small batch returned by the detection collate function.
            device_images = [
                image.to(
                    DEVICE,
                    non_blocking=True,
                )
                for image in images
            ]

            outputs = model(
                device_images
            )

            for output, target in zip(
                outputs,
                targets,
            ):

                scores = (
                    output[
                        "scores"
                    ]
                    .detach()
                    .float()
                    .cpu()
                    .numpy()
                )

                # Retain only detections relevant to at least one threshold.
                minimum_keep = (
                    scores
                    >= MINIMUM_SCORE_TO_RETAIN
                )

                retained_scores = scores[
                    minimum_keep
                ]

                if minimum_keep.any():

                    retained_probability_masks = (
                        output[
                            "masks"
                        ][
                            minimum_keep
                        ]
                        .detach()
                        .float()
                        .cpu()
                        .numpy()[
                            :,
                            0,
                        ]
                    )

                    retained_binary_masks = (
                        retained_probability_masks
                        >= mask_threshold
                    )

                    del retained_probability_masks

                else:

                    retained_binary_masks = np.zeros(
                        (
                            0,
                            PATCH_SIZE,
                            PATCH_SIZE,
                        ),
                        dtype=bool,
                    )

                reference_masks = (
                    target[
                        "masks"
                    ]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(
                        bool,
                        copy=False,
                    )
                )

                n_references = len(
                    reference_masks
                )

                # Compute IoU once for every retained detection.
                full_iou_matrix = (
                    compute_pairwise_mask_iou(
                        retained_binary_masks,
                        reference_masks,
                    )
                )

                for score_threshold in score_thresholds:

                    threshold_keep = (
                        retained_scores
                        >= score_threshold
                    )

                    threshold_prediction_count = int(
                        threshold_keep.sum()
                    )

                    threshold_iou_matrix = (
                        full_iou_matrix[
                            threshold_keep,
                            :
                        ]
                    )

                    for iou_threshold in iou_thresholds:

                        (
                            n_matches,
                            matched_ious,
                        ) = greedy_match_from_iou_matrix(
                            threshold_iou_matrix,
                            iou_threshold,
                        )

                        key = (
                            float(
                                score_threshold
                            ),
                            float(
                                iou_threshold
                            ),
                        )

                        counter = counters[
                            key
                        ]

                        counter[
                            "true_positive_crowns"
                        ] += n_matches

                        counter[
                            "false_positive_crowns"
                        ] += (
                            threshold_prediction_count
                            - n_matches
                        )

                        counter[
                            "false_negative_crowns"
                        ] += (
                            n_references
                            - n_matches
                        )

                        counter[
                            "n_reference_crowns"
                        ] += n_references

                        counter[
                            "n_predicted_crowns"
                        ] += (
                            threshold_prediction_count
                        )

                        counter[
                            "matched_iou_sum"
                        ] += float(
                            np.sum(
                                matched_ious
                            )
                        )

                        counter[
                            "matched_iou_count"
                        ] += len(
                            matched_ious
                        )

                        # This list contains only one value per matched crown,
                        # not full-resolution masks, so it remains small.
                        counter[
                            "matched_ious"
                        ].extend(
                            matched_ious
                        )

                # Explicitly discard all per-patch arrays.
                del scores
                del retained_scores
                del retained_binary_masks
                del reference_masks
                del full_iou_matrix
                del output

            del outputs
            del device_images

            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

            gc.collect()

    return counters


# ----------------------------------------------------------------------
# CONVERT COUNTERS TO A TABLE
# ----------------------------------------------------------------------

def counters_to_table(
    counters,
    split_name,
):

    rows = []

    for (
        score_threshold,
        iou_threshold,
    ), counter in counters.items():

        true_positive = counter[
            "true_positive_crowns"
        ]

        false_positive = counter[
            "false_positive_crowns"
        ]

        false_negative = counter[
            "false_negative_crowns"
        ]

        precision = (
            true_positive
            / max(
                true_positive
                + false_positive,
                1,
            )
        )

        recall = (
            true_positive
            / max(
                true_positive
                + false_negative,
                1,
            )
        )

        f1 = (
            2
            * precision
            * recall
            / max(
                precision
                + recall,
                1e-12,
            )
        )

        matched_ious = counter[
            "matched_ious"
        ]

        rows.append(
            {
                "split": split_name,
                "score_threshold": (
                    score_threshold
                ),
                "mask_threshold": (
                    MASK_THRESHOLD
                ),
                "iou_match_threshold": (
                    iou_threshold
                ),
                "n_reference_crowns": counter[
                    "n_reference_crowns"
                ],
                "n_predicted_crowns": counter[
                    "n_predicted_crowns"
                ],
                "true_positive_crowns": (
                    true_positive
                ),
                "false_positive_crowns": (
                    false_positive
                ),
                "false_negative_crowns": (
                    false_negative
                ),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "mean_matched_iou": (
                    counter[
                        "matched_iou_sum"
                    ]
                    / max(
                        counter[
                            "matched_iou_count"
                        ],
                        1,
                    )
                ),
                "median_matched_iou": (
                    float(
                        np.median(
                            matched_ious
                        )
                    )
                    if matched_ious
                    else 0.0
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ----------------------------------------------------------------------
# VALIDATION THRESHOLD SWEEP
# ----------------------------------------------------------------------

print("\nRunning validation threshold sweep...")

validation_counters = (
    stream_evaluate_loader(
        model=model,
        loader=validation_loader,
        score_thresholds=(
            SCORE_THRESHOLDS
        ),
        iou_thresholds=(
            IOU_THRESHOLDS
        ),
        mask_threshold=(
            MASK_THRESHOLD
        ),
        split_name="validation",
    )
)

validation_sweep = (
    counters_to_table(
        validation_counters,
        split_name="validation",
    )
)

selection_table = (
    validation_sweep[
        validation_sweep[
            "iou_match_threshold"
        ]
        == SELECTION_IOU_THRESHOLD
    ]
    .sort_values(
        [
            "f1",
            "recall",
            "precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

BEST_SCORE_THRESHOLD = float(
    selection_table.loc[
        0,
        "score_threshold",
    ]
)


print("\n" + "-" * 100)
print("VALIDATION SCORE-THRESHOLD SWEEP")
print("-" * 100)

print(
    "Selected score threshold:",
    BEST_SCORE_THRESHOLD,
)

print(
    "Selection IoU threshold:",
    SELECTION_IOU_THRESHOLD,
)

display(
    validation_sweep[
        [
            "iou_match_threshold",
            "score_threshold",
            "n_reference_crowns",
            "n_predicted_crowns",
            "precision",
            "recall",
            "f1",
            "mean_matched_iou",
            "median_matched_iou",
        ]
    ]
    .sort_values(
        [
            "iou_match_threshold",
            "score_threshold",
        ]
    )
    .style.format(
        {
            "iou_match_threshold": "{:.2f}",
            "score_threshold": "{:.2f}",
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1": "{:.4f}",
            "mean_matched_iou": "{:.4f}",
            "median_matched_iou": "{:.4f}",
        }
    )
)


# ----------------------------------------------------------------------
# FREE VALIDATION COUNTERS BEFORE TEST INFERENCE
# ----------------------------------------------------------------------

del validation_counters

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ----------------------------------------------------------------------
# INDEPENDENT TEST EVALUATION
#
# Only the selected score threshold is evaluated on test data.
# ----------------------------------------------------------------------

print("\nRunning independent test evaluation...")

test_counters = stream_evaluate_loader(
    model=model,
    loader=test_loader,
    score_thresholds=[
        BEST_SCORE_THRESHOLD
    ],
    iou_thresholds=IOU_THRESHOLDS,
    mask_threshold=MASK_THRESHOLD,
    split_name="test",
)

test_metrics = counters_to_table(
    test_counters,
    split_name="test",
)


print("\n" + "-" * 100)
print("INDEPENDENT TEST PERFORMANCE")
print("-" * 100)

print(
    "Validation-selected score threshold:",
    BEST_SCORE_THRESHOLD,
)

display(
    test_metrics.style.format(
        {
            "score_threshold": "{:.2f}",
            "mask_threshold": "{:.2f}",
            "iou_match_threshold": "{:.2f}",
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1": "{:.4f}",
            "mean_matched_iou": "{:.4f}",
            "median_matched_iou": "{:.4f}",
        }
    )
)


# ----------------------------------------------------------------------
# SAVE OUTPUTS
# ----------------------------------------------------------------------

VALIDATION_SWEEP_PATH = (
    MODEL_DIR
    / (
        "validation_score_threshold_sweep_"
        f"{LABEL_MODE.lower()}_"
        f"{'quick' if QUICK_TEST else 'full'}"
        "_memory_safe.csv"
    )
)

TEST_METRICS_PATH = (
    MODEL_DIR
    / (
        "test_instance_metrics_"
        "validation_selected_threshold_"
        f"{LABEL_MODE.lower()}_"
        f"{'quick' if QUICK_TEST else 'full'}"
        "_memory_safe.csv"
    )
)

THRESHOLD_SUMMARY_PATH = (
    MODEL_DIR
    / (
        "selected_instance_threshold_"
        f"{LABEL_MODE.lower()}_"
        f"{'quick' if QUICK_TEST else 'full'}"
        "_memory_safe.json"
    )
)

validation_sweep.to_csv(
    VALIDATION_SWEEP_PATH,
    index=False,
)

test_metrics.to_csv(
    TEST_METRICS_PATH,
    index=False,
)

threshold_summary = {
    "label_mode": LABEL_MODE,
    "quick_test": bool(
        QUICK_TEST
    ),
    "selected_score_threshold": (
        BEST_SCORE_THRESHOLD
    ),
    "mask_threshold": (
        MASK_THRESHOLD
    ),
    "selection_iou_threshold": (
        SELECTION_IOU_THRESHOLD
    ),
    "validation_best_metrics": (
        selection_table.iloc[
            0
        ].to_dict()
    ),
    "test_metrics": (
        test_metrics.to_dict(
            orient="records"
        )
    ),
}

THRESHOLD_SUMMARY_PATH.write_text(
    json.dumps(
        threshold_summary,
        indent=2,
    )
)

print("\nSaved:")
print(VALIDATION_SWEEP_PATH)
print(TEST_METRICS_PATH)
print(THRESHOLD_SUMMARY_PATH)


# ----------------------------------------------------------------------
# FINAL CLEANUP
# ----------------------------------------------------------------------

del test_counters

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nMemory-safe evaluation complete.")


BLOCK 14 — MEMORY-SAFE VALIDATION THRESHOLD SWEEP AND TEST EVALUATION

Running validation threshold sweep...


validation: streaming inference:   0%|          | 0/138 [00:00<?, ?it/s]


----------------------------------------------------------------------------------------------------
VALIDATION SCORE-THRESHOLD SWEEP
----------------------------------------------------------------------------------------------------
Selected score threshold: 0.5
Selection IoU threshold: 0.5


,iou_match_threshold,score_threshold,n_reference_crowns,n_predicted_crowns,precision,recall,f1,mean_matched_iou,median_matched_iou
0,0.25,0.05,770,12062,0.0496,0.7766,0.0932,0.5714,0.5939
2,0.25,0.10,770,6003,0.0858,0.6688,0.1521,0.5645,0.5821
4,0.25,0.15,770,3498,0.1272,0.5779,0.2085,0.5575,0.5820
6,0.25,0.20,770,2344,0.1685,0.5130,0.2537,0.5598,0.5844
8,0.25,0.25,770,1705,0.2023,0.4481,0.2788,0.5655,0.5955
10,0.25,0.30,770,1331,0.2374,0.4104,0.3008,0.5673,0.5976
12,0.25,0.40,770,860,0.2977,0.3325,0.3141,0.5838,0.6224
14,0.25,0.50,770,555,0.3640,0.2623,0.3049,0.6026,0.6387
16,0.25,0.60,770,335,0.4478,0.1948,0.2715,0.6228,0.6498
18,0.25,0.70,770,169,0.5917,0.1299,0.2130,0.6485,0.6848



Running independent test evaluation...


test: streaming inference:   0%|          | 0/198 [00:00<?, ?it/s]


----------------------------------------------------------------------------------------------------
INDEPENDENT TEST PERFORMANCE
----------------------------------------------------------------------------------------------------
Validation-selected score threshold: 0.5


,split,score_threshold,mask_threshold,iou_match_threshold,n_reference_crowns,n_predicted_crowns,true_positive_crowns,false_positive_crowns,false_negative_crowns,precision,recall,f1,mean_matched_iou,median_matched_iou
0,test,0.50,0.50,0.25,1499,645,268,377,1231,0.4155,0.1788,0.2500,0.6264,0.6853
1,test,0.50,0.50,0.50,1499,645,206,439,1293,0.3194,0.1374,0.1922,0.7033,0.7058



Saved:
/content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga/validation_score_threshold_sweep_chm_expanded_full_memory_safe.csv
/content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga/test_instance_metrics_validation_selected_threshold_chm_expanded_full_memory_safe.csv
/content/drive/MyDrive/Colab Notebooks/Musanga/work/mask_rcnn_non_musanga/selected_instance_threshold_chm_expanded_full_memory_safe.json

Memory-safe evaluation complete.


## BLOCK 15 — Visual prediction examples

In [15]:

# ======================================================================================
# BLOCK 15 — VISUAL PREDICTION EXAMPLES
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 15 — VISUAL PREDICTION EXAMPLES")
print("=" * 100)

N_EXAMPLES = 8

example_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

model.eval()

with torch.inference_mode():

    for example_index, (
        images,
        targets,
    ) in enumerate(
        example_loader,
        start=1,
    ):

        image = images[0]
        target = targets[0]

        output = model(
            [
                image.to(
                    DEVICE
                )
            ]
        )[0]

        keep = (
            output["scores"]
            .detach()
            .cpu()
            >= SCORE_THRESHOLD
        )

        predicted_masks = (
            output["masks"]
            .detach()
            .cpu()[
                keep,
                0,
            ]
            >= MASK_THRESHOLD
        )

        rgb = np.moveaxis(
            image[:3].numpy(),
            0,
            -1,
        )

        observed_overlay = rgb.copy()
        predicted_overlay = rgb.copy()

        for mask in target["masks"]:
            boundary = morphology.binary_dilation(
                mask.numpy().astype(bool),
                morphology.disk(1),
            ) ^ morphology.binary_erosion(
                mask.numpy().astype(bool),
                morphology.disk(1),
            )

            observed_overlay[
                boundary
            ] = np.array(
                [1.0, 0.8, 0.0]
            )

        for mask in predicted_masks:
            mask_np = mask.numpy()

            boundary = morphology.binary_dilation(
                mask_np,
                morphology.disk(1),
            ) ^ morphology.binary_erosion(
                mask_np,
                morphology.disk(1),
            )

            predicted_overlay[
                boundary
            ] = np.array(
                [1.0, 0.1, 0.1]
            )

        figure, axes = plt.subplots(
            1,
            3,
            figsize=(15, 5),
        )

        axes[0].imshow(rgb)
        axes[0].set_title("RGB")

        axes[1].imshow(
            observed_overlay
        )
        axes[1].set_title(
            f"Observed crowns ({len(target['masks'])})"
        )

        axes[2].imshow(
            predicted_overlay
        )
        axes[2].set_title(
            f"Predicted crowns ({len(predicted_masks)})"
        )

        for axis in axes:
            axis.axis("off")

        figure.tight_layout()
        plt.show()
        plt.close(figure)

        if example_index >= N_EXAMPLES:
            break


Output hidden; open in https://colab.research.google.com to view.

## BLOCK 16 — Notes before full training

After the quick test:

1. inspect the visual examples;
2. check whether expanded labels better approximate complete crowns;
3. set `QUICK_TEST = False`;
4. keep `LABEL_MODE = "CHM_EXPANDED"` only if the expansion outputs look credible;
5. rerun from the top.

For a publication, compare at least:

- original conservative labels;
- CHM-expanded labels;
- watershed baseline;
- Mask R-CNN predictions.

Whole-orthomosaic inference should only be added after the model performs acceptably on the spatially independent test set.
